In [ ]:
from pathlib import Path
from collections import defaultdict
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = "/Users/gupta/Documents/DIS-IND/data/btc"
CHUNK_SIZE = 200_000


# ============================================================
# ANALYZER
# ============================================================

def analyze_dataset(dataset_path, chunk_size=100_000):

    dataset_path = Path(dataset_path)

    # --------------------------------------------------------
    # Find files
    # --------------------------------------------------------

    csv_files = sorted(dataset_path.glob("*.csv"))
    tbl_files = sorted(dataset_path.glob("*.tbl"))

    if csv_files and tbl_files:
        raise ValueError(
            "Dataset contains both .csv and .tbl files. "
            "Use only one format per dataset."
        )

    if csv_files:
        files = csv_files
        file_type = "csv"
        separator = ","

    elif tbl_files:
        files = tbl_files
        file_type = "tbl"
        separator = "|"

    else:
        raise FileNotFoundError(
            f"No .csv or .tbl files found in {dataset_path}"
        )


    # ========================================================
    # GLOBAL STATISTICS
    # ========================================================

    total_rows = 0
    max_rows_per_table = 0
    total_attributes = 0

    all_attribute_distinct_counts = []

    # All unique values across the complete dataset
    all_dataset_values = set()

    # value -> set of attributes containing the value
    value_to_attributes = defaultdict(set)

    table_reports = []


    # --------------------------------------------------------
    # Dataset size
    # --------------------------------------------------------

    total_size_bytes = sum(
        file.stat().st_size
        for file in files
    )

    total_size_mb = total_size_bytes / (1024 * 1024)


    # ========================================================
    # PROCESS TABLES
    # ========================================================

    for file_path in files:

        print(f"Analyzing: {file_path.name}")

        table_rows = 0
        table_size_mb = file_path.stat().st_size / (1024 * 1024)


        # ----------------------------------------------------
        # Get columns
        # ----------------------------------------------------

        if file_type == "csv":

            header = pd.read_csv(
                file_path,
                nrows=0
            )

            columns = list(header.columns)

        else:

            sample = pd.read_csv(
                file_path,
                sep=separator,
                header=None,
                nrows=1,
                dtype=str
            )

            # Remove trailing empty TBL column
            if (
                len(sample.columns) > 0
                and sample.iloc[:, -1].isna().all()
            ):
                number_of_columns = len(sample.columns) - 1

            else:
                number_of_columns = len(sample.columns)

            columns = [
                f"column_{i+1}"
                for i in range(number_of_columns)
            ]


        number_of_attributes = len(columns)

        total_attributes += number_of_attributes


        # ----------------------------------------------------
        # Distinct values per attribute
        # ----------------------------------------------------

        distinct_values = {
            column: set()
            for column in columns
        }


        # ----------------------------------------------------
        # Chunk reader
        # ----------------------------------------------------

        if file_type == "csv":

            reader = pd.read_csv(
                file_path,
                chunksize=chunk_size,
                dtype=str,
                keep_default_na=False
            )

        else:

            reader = pd.read_csv(
                file_path,
                sep=separator,
                header=None,
                chunksize=chunk_size,
                dtype=str,
                keep_default_na=False
            )


        # ----------------------------------------------------
        # Process chunks
        # ----------------------------------------------------

        for chunk in reader:

            if (
                file_type == "tbl"
                and len(chunk.columns) > len(columns)
            ):
                chunk = chunk.iloc[:, :len(columns)]

            chunk.columns = columns

            table_rows += len(chunk)


            for column in columns:

                unique_values = chunk[column].unique()

                # Distinct values in this attribute
                distinct_values[column].update(
                    unique_values
                )

                # Distinct values in entire dataset
                all_dataset_values.update(
                    unique_values
                )

                # Qualified attribute name
                # Example:
                # orders.customer_id
                qualified_attribute = (
                    f"{file_path.stem}.{column}"
                )

                # Record where every value appears
                for value in unique_values:

                    value_to_attributes[value].add(
                        qualified_attribute
                    )


        # ----------------------------------------------------
        # Table statistics
        # ----------------------------------------------------

        total_rows += table_rows

        max_rows_per_table = max(
            max_rows_per_table,
            table_rows
        )


        table_distinct_counts = []

        for column in columns:

            count = len(
                distinct_values[column]
            )

            table_distinct_counts.append(count)

            all_attribute_distinct_counts.append(count)


        if table_distinct_counts:

            table_max_distinct = max(
                table_distinct_counts
            )

            table_avg_distinct = (
                sum(table_distinct_counts)
                / len(table_distinct_counts)
            )

        else:

            table_max_distinct = 0
            table_avg_distinct = 0


        table_reports.append({

            "table":
                file_path.name,

            "size_mb":
                table_size_mb,

            "rows":
                table_rows,

            "attributes":
                number_of_attributes,

            "max_distinct":
                table_max_distinct,

            "average_distinct":
                table_avg_distinct
        })


    # ========================================================
    # DATASET-WIDE DISTINCT STATISTICS
    # ========================================================

    if all_attribute_distinct_counts:

        max_distinct_values_per_attribute = max(
            all_attribute_distinct_counts
        )

        average_distinct_values_per_attribute = (
            sum(all_attribute_distinct_counts)
            / len(all_attribute_distinct_counts)
        )

    else:

        max_distinct_values_per_attribute = 0
        average_distinct_values_per_attribute = 0


    total_distinct_values_dataset = len(
        all_dataset_values
    )


    # ========================================================
    # COUNT CLUSTERS ONLY
    # ========================================================
    #
    # Example:
    #
    # 1 -> {A,B,C}
    # 2 -> {A,B}
    # 3 -> {A,B,D}
    # 4 -> {A,B}
    #
    # Unique attribute sets:
    #
    # {A,B,C}
    # {A,B}
    # {A,B,D}
    #
    # Number of clusters = 3
    # ========================================================

    unique_attribute_sets = set()

    for attributes in value_to_attributes.values():

        unique_attribute_sets.add(
            frozenset(attributes)
        )


    number_of_clusters = len(
        unique_attribute_sets
    )


    # ========================================================
    # FINAL SUMMARY
    # ========================================================

    print()
    print("=" * 80)
    print("DATASET SUMMARY")
    print("=" * 80)

    print(
        f"Number of tables                       : "
        f"{len(files):,}"
    )

    print(
        f"Dataset size (MB)                      : "
        f"{total_size_mb:,.2f}"
    )

    print(
        f"Total rows                             : "
        f"{total_rows:,}"
    )

    print(
        f"Max # rows per table                   : "
        f"{max_rows_per_table:,}"
    )

    print(
        f"Total # attributes                     : "
        f"{total_attributes:,}"
    )

    print(
        f"Max # distinct values per attribute    : "
        f"{max_distinct_values_per_attribute:,}"
    )

    print(
        f"Average # distinct values per attribute: "
        f"{average_distinct_values_per_attribute:,.2f}"
    )

    print(
        f"Total distinct values in dataset       : "
        f"{total_distinct_values_dataset:,}"
    )

    print(
        f"Total # clusters                       : "
        f"{number_of_clusters:,}"
    )


    # ========================================================
    # TABLE SUMMARY
    # ========================================================

    print()
    print("=" * 80)
    print("TABLE SUMMARY")
    print("=" * 80)

    for table in table_reports:

        print(
            f"{table['table']:<35} "
            f"Rows={table['rows']:<12,} "
            f"Attributes={table['attributes']:<6,} "
            f"Size={table['size_mb']:>10,.2f} MB "
            f"MaxDistinct={table['max_distinct']:>12,} "
            f"AvgDistinct={table['average_distinct']:>12,.2f}"
        )


    # ========================================================
    # RETURN REPORT
    # ========================================================

    return {

        "number_of_tables":
            len(files),

        "dataset_size_mb":
            total_size_mb,

        "total_rows":
            total_rows,

        "max_rows_per_table":
            max_rows_per_table,

        "total_attributes":
            total_attributes,

        "max_distinct_values_per_attribute":
            max_distinct_values_per_attribute,

        "average_distinct_values_per_attribute":
            average_distinct_values_per_attribute,

        "total_distinct_values_dataset":
            total_distinct_values_dataset,

        "number_of_clusters":
            number_of_clusters,

        "tables":
            table_reports
    }


# ============================================================
# RUN
# ============================================================

report = analyze_dataset(
    DATASET_PATH,
    chunk_size=CHUNK_SIZE
)

Analyzing: 1m_BTC_2021.csv

DATASET SUMMARY
Number of tables                       : 1
Dataset size (MB)                      : 2,987.15
Total rows                             : 523,808
Max # rows per table                   : 523,808
Total # attributes                     : 375
Max # distinct values per attribute    : 523,808
Average # distinct values per attribute: 396,491.02
Total distinct values in dataset       : 122,459,624
Total # clusters                       : 63,782

TABLE SUMMARY
1m_BTC_2021.csv                     Rows=523,808      Attributes=375    Size=  2,987.15 MB MaxDistinct=     523,808 AvgDistinct=  396,491.02


: 